4. Agrégation temporelle  → features pour Prophet
5. Train Prophet          → forecast sur 7 jours
6. Log MLflow             → un run complet, les deux modèles enregistrés

---
## Imports initiaux

In [3]:
import pandas as pd
import os

# Logging
import logging
logger = logging.getLogger(__name__)

In [4]:
# Engine
from pipeline.core.src.sql import get_engine

engine = get_engine()

[INFO] pipeline_sql: Engine created — neondb_owner@ep-quiet-sun-al5bcj46-pooler.c-3.eu-central-1.aws.neon.tech/theguardian-sentiment


---
- Fetch les scores depuis PostgreSQL (articles avec sentiment_label NOT NULL)

In [5]:
# Fetch processed
from pipeline.core.src.sql import fetch_processed

articles = fetch_processed(engine=engine, schema="theguardian")
articles.shape

[INFO] pipeline_sql: Fetched 25888 already processed articles from theguardian.articles


(25888, 6)

In [6]:
articles

,id,date,text,sentiment_label,sentiment_score,created_at
0,business/2017/jan/01/hmrc-tax-evasion-enablers...,2017-01-01,HMRC empowered to name and shame tax evasion '...,negative,0.515169,2026-05-04 14:57:25.336067
1,business/2017/jan/01/alan-joyce-among-qantas-p...,2017-01-01,Alan Joyce among Qantas passengers stranded in...,negative,0.946898,2026-05-04 14:57:25.336067
2,world/2017/jan/01/chinese-manufacturing-purcha...,2017-01-01,Chinese manufacturing sector expands for fifth...,negative,0.971667,2026-05-04 14:57:25.336067
3,business/2017/jan/01/uk-economy-will-fare-bett...,2017-01-01,Why the UK economy could fare better in 2017 t...,negative,0.728727,2026-05-04 14:57:25.336067
4,business/2017/jan/01/observer-business-review-...,2017-01-01,Observer business agenda’s review of the year....,negative,0.698265,2026-05-04 14:57:25.336067
...,...,...,...,...,...,...
25883,business/2023/oct/31/eurozone-economy-shrinks-...,2023-10-31,"Eurozone economy shrinks by 0.1%, putting it a...",negative,0.967309,2026-05-04 15:25:49.587685
25884,business/live/2023/oct/31/eurozone-economy-fre...,2023-10-31,Eurozone on brink of recession as economy shri...,negative,0.968375,2026-05-04 15:25:49.587685
25885,business/2023/oct/31/sam-bankman-fried-courtro...,2023-10-31,Buff and hulking or small and meek? Courtroom ...,neutral,0.565189,2026-05-04 15:25:49.587685
25886,business/2023/oct/31/sam-bankman-fried-fraud-t...,2023-10-31,Sam Bankman-Fried wraps up testimony as fraud ...,negative,0.722512,2026-05-04 15:25:49.587685


---
- Agrégation hebdomadaire → moyenne des scores par semaine

In [7]:
# Mapper les sentiment_label par un nombre sur tous les articles pour pouvoir comparer les articles sur la même échelle
articles["sentiment_signed"] = articles["sentiment_score"] * articles["sentiment_label"].map({
    "positive":  1,
    "negative": -1,
    "neutral":   0
})

# Agrégation hebdomadaire
articles["date"] = pd.to_datetime(articles["date"])
weekly = (
    articles
    .set_index("date")
    .resample("W")["sentiment_signed"]  # regroupe par semaine, mieux que groupby pour série temporelle
    .mean()
    .reset_index()
    .rename(columns={"date": "ds", "sentiment_signed": "y"})
)
weekly

,ds,y
0,2017-01-01,-0.549060
1,2017-01-08,-0.394250
2,2017-01-15,-0.381709
3,2017-01-22,-0.415620
4,2017-01-29,-0.487076
...,...,...
353,2023-10-08,-0.470611
354,2023-10-15,-0.511382
355,2023-10-22,-0.466862
356,2023-10-29,-0.501358


---
### Train Prophet

In [8]:
from prophet import Prophet

In [9]:
# Fonction pour entraîner un modèle et faire des prédictions
def train_and_predict(df):
    """
    Train Prophet model on a DataFrame and return projection according to the tendancy over the past period.
    
    Args:
        df (pd.DataFrame): DataFrame with columns: 'ds' (date) and 'y' (target).
    
    Returns:
        pd.DataFrame: Predictions with 'ds', 'yhat', 'yhat_lower', 'yhat_upper', etc.
    """
    # Train model
    model = Prophet()
    model.fit(df)
    
    # Create a DataFrame for projections
    # Choose a window of projections (periods=)
    # Here 30 days
    future = model.make_future_dataframe(periods=30)
    
    # Predict
    forecast = model.predict(future)
    
    return model, forecast

In [10]:
model, forecast = train_and_predict(weekly)

[INFO] prophet: Disabling weekly seasonality. Run prophet with weekly_seasonality=True to override this.
[INFO] prophet: Disabling daily seasonality. Run prophet with daily_seasonality=True to override this.
[INFO] cmdstanpy: Chain [1] start processing
[INFO] cmdstanpy: Chain [1] done processing


In [11]:
forecast

,ds,trend,yhat_lower,yhat_upper,trend_lower,trend_upper,additive_terms,additive_terms_lower,additive_terms_upper,yearly,yearly_lower,yearly_upper,multiplicative_terms,multiplicative_terms_lower,multiplicative_terms_upper,yhat
0,2017-01-01,-0.459060,-0.554240,-0.314394,-0.459060,-0.459060,0.027900,0.027900,0.027900,0.027900,0.027900,0.027900,0.0,0.0,0.0,-0.431160
1,2017-01-08,-0.459555,-0.515542,-0.281165,-0.459555,-0.459555,0.057908,0.057908,0.057908,0.057908,0.057908,0.057908,0.0,0.0,0.0,-0.401647
2,2017-01-15,-0.460050,-0.506618,-0.283173,-0.460050,-0.460050,0.061637,0.061637,0.061637,0.061637,0.061637,0.061637,0.0,0.0,0.0,-0.398413
3,2017-01-22,-0.460545,-0.536583,-0.294317,-0.460545,-0.460545,0.042508,0.042508,0.042508,0.042508,0.042508,0.042508,0.0,0.0,0.0,-0.418037
4,2017-01-29,-0.461040,-0.547043,-0.316440,-0.461040,-0.461040,0.019532,0.019532,0.019532,0.019532,0.019532,0.019532,0.0,0.0,0.0,-0.441508
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
383,2023-12-01,-0.481257,-0.579852,-0.349814,-0.481268,-0.481244,0.021608,0.021608,0.021608,0.021608,0.021608,0.021608,0.0,0.0,0.0,-0.459649
384,2023-12-02,-0.481292,-0.575152,-0.345599,-0.481305,-0.481278,0.018555,0.018555,0.018555,0.018555,0.018555,0.018555,0.0,0.0,0.0,-0.462737
385,2023-12-03,-0.481327,-0.577588,-0.355019,-0.481342,-0.481307,0.015169,0.015169,0.015169,0.015169,0.015169,0.015169,0.0,0.0,0.0,-0.466158
386,2023-12-04,-0.481363,-0.593427,-0.359124,-0.481384,-0.481334,0.011502,0.011502,0.011502,0.011502,0.011502,0.011502,0.0,0.0,0.0,-0.469860


In [12]:
forecast.columns

Index(['ds', 'trend', 'yhat_lower', 'yhat_upper', 'trend_lower', 'trend_upper',
       'additive_terms', 'additive_terms_lower', 'additive_terms_upper',
       'yearly', 'yearly_lower', 'yearly_upper', 'multiplicative_terms',
       'multiplicative_terms_lower', 'multiplicative_terms_upper', 'yhat'],
      dtype='object')

In [13]:
from prophet.plot import plot_plotly, plot_components_plotly

# Plot principal : historique + prévision
fig1 = plot_plotly(model, forecast)
fig1.show()

# Plot des composantes : tendance + saisonnalité
fig2 = plot_components_plotly(model, forecast)
fig2.show()

Chute jusqu'en 2020 et remontée

Le sentiment de la presse économique pendant le covid a été massivement négatif au moment du choc (mars-avril 2020), puis les marchés ont rebondi très vite — le S&P500 a récupéré ses pertes en 6 mois. La presse économique suit les marchés plus que la réalité sanitaire. Les plans de relance massifs, les vaccins annoncés fin 2020, la réouverture anticipée des économies — tout ça génère du sentiment positif dans la presse business même en plein lockdown.
C'est une validation indirecte que FinBERT capte bien le sentiment économique médiatique — pas le sentiment sanitaire ou social.

Sur la saisonnalité

Le pattern identifié :
vendre mi-janvier      → fin du rally de Noël
acheter fin mars       → creux de début d'année
vendre début mai       → "Sell in May"
racheter 15 août       → creux estival, avant le rally de rentrée
C'est aligné avec les cycles boursiers classiques. Le modèle le retrouve à partir d'articles de presse économique sans aucune donnée de marché en entrée.

---
### Logging MLFlow

In [1]:
import mlflow
from mlflow import MlflowClient
import mlflow.prophet
from mlflow.models import infer_signature

In [14]:
mlflow.set_tracking_uri(os.getenv("MLFLOW_TRACKING_URI"))
mlflow.set_experiment("theguardian-sentiment")

2026/05/05 11:20:16 INFO mlflow.tracking.fluent: Experiment with name 'theguardian-sentiment' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/1', creation_time=1777972816257, experiment_id='1', last_update_time=1777972816257, lifecycle_stage='active', name='theguardian-sentiment', tags={}, trace_location=None, workspace='default'>

In [15]:
with mlflow.start_run(run_name="prophet-weekly-sentiment"):
    # Params
    mlflow.log_param("model", os.getenv("MODEL_NAME"))
    mlflow.log_param("granularity", "weekly")
    mlflow.log_param("horizon_days", 30)
    mlflow.log_param("history_start", str(weekly["ds"].min()))
    mlflow.log_param("history_end", str(weekly["ds"].max()))

    # Métriques sur le fit historique
    from sklearn.metrics import mean_absolute_error
    fitted = forecast.loc[forecast["ds"].isin(weekly["ds"]), "yhat"]
    mae = mean_absolute_error(weekly["y"], fitted)
    mlflow.log_metric("mae", mae)

    # Modèle
    mlflow.prophet.log_model(model, name=os.getenv("MODEL_NAME"))

    run_id = mlflow.active_run().info.run_id
    logger.info("MLflow run logged — run_id: %s", run_id)

[INFO] __main__: MLflow run logged — run_id: 1aac7f465aea4fe4a439c2b262a5a741


🏃 View run prophet-weekly-sentiment at: https://fabien1974-mlflow-server-theguardian-sentiment.hf.space/#/experiments/1/runs/1aac7f465aea4fe4a439c2b262a5a741
🧪 View experiment at: https://fabien1974-mlflow-server-theguardian-sentiment.hf.space/#/experiments/1


In [16]:
from mlflow import MlflowClient

client = MlflowClient()

# Enregistrer le modèle dans le registry
model_name = os.getenv('MODEL_NAME')
model_uri = f"runs:/{run_id}/{model_name}"
mv = mlflow.register_model(model_uri, name=model_name)

# Assigner l'alias @production
client.set_registered_model_alias(
    name=model_name,
    alias="production",
    version=mv.version
)

logger.info("Model v%s promoted to @production", mv.version)

Successfully registered model 'prophet'.
2026/05/05 11:42:07 WARNING mlflow.tracking._model_registry.fluent: Run with id 1aac7f465aea4fe4a439c2b262a5a741 has no artifacts at artifact path 'prophet', registering model based on models:/m-9b047ea722bb419ebc207405727263a8 instead
2026/05/05 11:42:12 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: prophet, version 1
Created version '1' of model 'prophet'.
[INFO] __main__: Model v1 promoted to @production


---
# ONE SHOT
---
### Ajout de la table forecasts
#### Cette table devrait être créée au moment d'initdb mais elle a été conçue une fois que la base est déjà constituée et on ne veut pas drop les autres tables, donc on créé forecasts individuellement

In [1]:
from pipeline.core.src.sql import get_engine, create_table_forecasts

engine = get_engine()
create_table_forecasts(engine)

[INFO] pipeline_sql: Engine created — neondb_owner@ep-quiet-sun-al5bcj46-pooler.c-3.eu-central-1.aws.neon.tech/theguardian-sentiment
[INFO] pipeline_sql: Table ready: theguardian.forecasts
